# 📊 Final Phase — Notebook 2: Full Evaluation Suite
## Quantum vs Classical — Complete Metric Comparison

**What this notebook computes:**
- ✅ Character Error Rate (CER) on held-out test set
- ✅ Word Error Rate / Exact-Plate Accuracy (WER)
- ✅ Inference speed (ms per image)
- ✅ Parameter count comparison
- ✅ 10-sample side-by-side prediction panel
- ✅ Final comparison table (for your report)

> **Prerequisites:** Run Notebook 1 first to generate both model checkpoints.

## Cell 1 — Setup (copy from Notebook 1)

In [ ]:
!pip install pennylane pennylane-lightning jiwer editdistance -q

import os, json, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pennylane as qml
import editdistance

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')

from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Configuration & Dataset

In [ ]:
PROJECT_PATH   = '/content/drive/MyDrive/Quantum_LPR_Project'
CHECKPOINT_DIR = os.path.join(PROJECT_PATH, 'checkpoints')
CSV_PATH       = '/content/drive/MyDrive/License Plate/Manifests/2_train_hr_images.csv'
ZIP_PATH       = '/content/drive/MyDrive/License Plate/wYe7pBJ7-train.zip'
EXTRACT_PATH   = '/content/lpr_train'
BATCH_SIZE     = 16
SEED           = 42

CHARS    = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i+1: c for i, c in enumerate(CHARS)}

# Extract dataset if not present
import zipfile
if not os.path.exists(os.path.join(EXTRACT_PATH, 'train')):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print('✅ Dataset extracted.')

# Dataset class
class LPRDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, mode='eval'):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir   = root_dir
        self.transform  = transform
        self.mode       = mode

    def __len__(self):
        return len(self.data_frame)

    def encode_text(self, text):
        return [CHAR2IDX[c] for c in str(text).upper() if c in CHAR2IDX]

    def make_night(self, img_tensor):
        if torch.rand(1) < 0.5:
            gamma = np.random.uniform(2.0, 3.5)
            img_tensor = torch.pow(img_tensor, gamma)
            img_tensor = torch.clamp(img_tensor + torch.randn_like(img_tensor)*0.05, 0, 1)
        return img_tensor

    def __getitem__(self, idx):
        img_path   = self.data_frame.iloc[idx]['path']
        label_text = self.data_frame.iloc[idx]['label']
        full_path  = img_path if img_path.startswith('/content') \
                     else os.path.join(self.root_dir, img_path)
        try:
            image = Image.open(full_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (256, 64))
        if self.transform:
            image = self.transform(image)
        if self.mode == 'night':        # Night simulation for test demo
            image = self.make_night(image)
        encoded = torch.tensor(self.encode_text(label_text), dtype=torch.long)
        length  = torch.tensor(len(encoded), dtype=torch.long)
        return image, encoded, length


def collate_fn(batch):
    images, labels, lengths = zip(*batch)
    return torch.stack(images,0), torch.cat(labels), torch.stack(lengths,0)


transform = transforms.Compose([transforms.Resize((64,256)), transforms.ToTensor()])

# Rebuild test set (same seed as Notebook 1)
from torch.utils.data import random_split
torch.manual_seed(SEED)
full_ds  = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='eval')
N        = len(full_ds)
n_train, n_val = int(N*0.70), int(N*0.15)
n_test   = N - n_train - n_val
_, _, test_set = random_split(full_ds, [n_train, n_val, n_test])
test_loader = DataLoader(test_set, BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

# Night test set (for demo visualization)
night_ds = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='night')
night_test = Subset(night_ds, test_set.indices)
night_loader = DataLoader(night_test, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'✅ Test set size: {n_test} images')

## Cell 3 — Load Both Models

In [ ]:
# ── Model definitions (re-paste from Notebook 1) ──────────
N_QUBITS, N_LAYERS = 8, 2
dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch')
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class ZeroDCE_Light(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,24,3,1,1), nn.Tanh())
    def forward(self, x):
        A, e = self.conv(x), x
        for i in range(8): e = e + A[:,3*i:3*(i+1)] * (torch.pow(e,2) - e)
        return e

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit,
                                           {'weights': (N_LAYERS, N_QUBITS, 3)})
    def forward(self, x): return self.q_layer(x)

class HybridLPRNet_8Q(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.quantum   = QuantumLayer()
        self.rnn       = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier= nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b,c,h,w = x.size()
        x = x.mean(dim=2).permute(0,2,1)
        q = self.quantum(x.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)
        return self.classifier(self.rnn(q)[0]).permute(1,0,2)

class ClassicalLPRNet(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.classical = nn.Sequential(nn.Linear(N_QUBITS,16),nn.Tanh(),nn.Linear(16,N_QUBITS))
        self.rnn       = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier= nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b,c,h,w = x.size()
        x = self.classical(x.mean(dim=2).permute(0,2,1))
        return self.classifier(self.rnn(x)[0]).permute(1,0,2)

# ── Load checkpoints ────────────────────────────────────────
def load_best(model, best_file, fallback_file):
    for fname in [best_file, fallback_file]:
        path = os.path.join(CHECKPOINT_DIR, fname)
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=device)
            model.load_state_dict(ckpt['model_state_dict'])
            ep  = ckpt.get('epoch', '?')
            cer = ckpt.get('val_cer', '?')
            print(f'  ✅ Loaded {fname}  (Epoch {ep}, val_CER={cer})')
            return
    print(f'  ❌ No checkpoint found for {best_file}/{fallback_file}')

q_model = HybridLPRNet_8Q(37).to(device)
c_model = ClassicalLPRNet(37).to(device)

print('Loading Quantum model ...')
load_best(q_model, '8qubit_best.pth', '8qubit_model.pth')
print('Loading Classical model ...')
load_best(c_model, 'classical_best.pth', 'classical_model.pth')

q_model.eval()
c_model.eval()
print('\n✅ Both models ready.')

## Cell 4 — CTC Decode & Metric Functions

In [ ]:
def ctc_greedy_decode(log_probs):
    indices = torch.argmax(log_probs, dim=2)
    batch = []
    for b in range(indices.size(1)):
        seq, chars, prev = indices[:,b].tolist(), [], -1
        for idx in seq:
            if idx != 0 and idx != prev: chars.append(IDX2CHAR.get(idx,''))
            prev = idx
        batch.append(''.join(chars))
    return batch

def decode_targets(targets, lengths):
    strings, offset = [], 0
    for l in lengths.tolist():
        strings.append(''.join(IDX2CHAR.get(i,'') for i in targets[offset:offset+l].tolist()))
        offset += l
    return strings

def evaluate_full(model, loader, model_name='Model'):
    """Returns dict with cer, wer, total_samples, and per-sample results."""
    model.eval()
    total_cer, total_wer, count = 0.0, 0, 0
    all_preds, all_truths = [], []

    with torch.no_grad():
        for imgs, targets, lengths in loader:
            preds   = model(imgs.to(device)).cpu()
            decoded = ctc_greedy_decode(preds)
            actuals = decode_targets(targets, lengths)
            for pred, true in zip(decoded, actuals):
                if len(true) > 0:
                    total_cer += editdistance.eval(pred, true) / len(true)
                total_wer += (0 if pred == true else 1)
                count += 1
                all_preds.append(pred)
                all_truths.append(true)

    cer = total_cer / count if count else 1.0
    wer = total_wer / count if count else 1.0
    print(f'  [{model_name}] CER={cer:.4f} ({cer*100:.1f}%)  '
          f'WER={wer:.4f} ({wer*100:.1f}%)  Accuracy={((1-wer)*100):.1f}%')
    return {'cer': cer, 'wer': wer, 'n': count,
            'preds': all_preds, 'truths': all_truths}

print('✅ Metric functions ready.')

## Cell 5 — Run Full Evaluation on Test Set

In [ ]:
print('\n📊 Evaluating on CLEAN test set (no night augmentation) ...')
print('─' * 60)
q_results_clean = evaluate_full(q_model, test_loader,  'Quantum  ⚡')
c_results_clean = evaluate_full(c_model, test_loader,  'Classical 🔷')

print('\n📊 Evaluating on NIGHT test set (night augmentation applied) ...')
print('─' * 60)
q_results_night = evaluate_full(q_model, night_loader, 'Quantum  ⚡')
c_results_night = evaluate_full(c_model, night_loader, 'Classical 🔷')

## Cell 6 — Inference Speed Benchmark

In [ ]:
def benchmark_speed(model, model_name, n_runs=50):
    """Measure average inference time in ms per image."""
    dummy = torch.randn(1, 3, 64, 256).to(device)
    model.eval()

    # Warm-up
    for _ in range(5):
        with torch.no_grad(): model(dummy)

    times = []
    for _ in range(n_runs):
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad(): model(dummy)
        if device.type == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)  # ms

    mean_ms = np.mean(times)
    std_ms  = np.std(times)
    print(f'  [{model_name}] {mean_ms:.2f} ± {std_ms:.2f} ms/image')
    return mean_ms, std_ms

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('⏱️  Speed benchmark (single image inference, GPU) ...')
q_speed, q_std = benchmark_speed(q_model, 'Quantum  ⚡')
c_speed, c_std = benchmark_speed(c_model, 'Classical 🔷')

q_params = count_params(q_model)
c_params = count_params(c_model)

## Cell 7 — Final Comparison Table (for your Report!)

In [ ]:
import pandas as pd

table = pd.DataFrame({
    'Metric': [
        'CER — Clean Images',
        'WER — Clean Images',
        'Plate Accuracy — Clean',
        'CER — Night Images',
        'WER — Night Images',
        'Plate Accuracy — Night',
        'Inference Speed (ms/img)',
        'Trainable Parameters',
    ],
    'Quantum ⚡': [
        f"{q_results_clean['cer']*100:.1f}%",
        f"{q_results_clean['wer']*100:.1f}%",
        f"{(1-q_results_clean['wer'])*100:.1f}%",
        f"{q_results_night['cer']*100:.1f}%",
        f"{q_results_night['wer']*100:.1f}%",
        f"{(1-q_results_night['wer'])*100:.1f}%",
        f"{q_speed:.1f} ± {q_std:.1f}",
        f"{q_params:,}",
    ],
    'Classical 🔷': [
        f"{c_results_clean['cer']*100:.1f}%",
        f"{c_results_clean['wer']*100:.1f}%",
        f"{(1-c_results_clean['wer'])*100:.1f}%",
        f"{c_results_night['cer']*100:.1f}%",
        f"{c_results_night['wer']*100:.1f}%",
        f"{(1-c_results_night['wer'])*100:.1f}%",
        f"{c_speed:.1f} ± {c_std:.1f}",
        f"{c_params:,}",
    ]
})

# Determine winner for each row
def highlight_winner(row):
    return ['background-color: #c8f7c5' if i == 1 else
            'background-color: #f7c8c5' if i == 2 else ''
            for i in range(len(row))]

print('\n' + '='*65)
print('  FINAL COMPARISON TABLE  (lower CER/WER = better)')
print('='*65)
print(table.to_string(index=False))
print('='*65)

# Also save as CSV for the report
table.to_csv(os.path.join(PROJECT_PATH, 'final_comparison_table.csv'), index=False)
print('\n✅ Saved comparison table → final_comparison_table.csv')

## Cell 8 — Bar Chart: Quantum vs Classical

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Quantum vs Classical — LPR Performance', fontsize=15, fontweight='bold')

colors_q = '#7B2D8B'   # Quantum — purple
colors_c = '#2196F3'   # Classical — blue

x = np.arange(2)
w = 0.35

# Left: CER on clean vs night
ax = axes[0]
ax.bar(x - w/2, [q_results_clean['cer']*100, q_results_night['cer']*100],
       w, label='Quantum ⚡', color=colors_q, alpha=0.85)
ax.bar(x + w/2, [c_results_clean['cer']*100, c_results_night['cer']*100],
       w, label='Classical 🔷', color=colors_c, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(['Clean Images', 'Night Images'])
ax.set_ylabel('CER (%) — lower is better')
ax.set_title('Character Error Rate')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Right: Plate Accuracy
ax = axes[1]
ax.bar(x - w/2, [(1-q_results_clean['wer'])*100, (1-q_results_night['wer'])*100],
       w, label='Quantum ⚡', color=colors_q, alpha=0.85)
ax.bar(x + w/2, [(1-c_results_clean['wer'])*100, (1-c_results_night['wer'])*100],
       w, label='Classical 🔷', color=colors_c, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(['Clean Images', 'Night Images'])
ax.set_ylabel('Accuracy (%) — higher is better')
ax.set_title('Exact Plate Match Accuracy')
ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(PROJECT_PATH, 'comparison_bar_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved comparison_bar_chart.png')

## Cell 9 — 10-Sample Side-by-Side Prediction Panel

In [ ]:
def show_sample_predictions(q_model, c_model, dataset, n=10):
    """
    For each of n random test samples, show:
      Col 1: Original clean image
      Col 2: Night-augmented input
      Col 3: ZeroDCE enhanced output
      Col 4: Quantum prediction (✅/❌)
      Col 5: Classical prediction (✅/❌)
    """
    clean_ds = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='eval')
    night_ds  = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='night')

    indices = np.random.choice(test_set.indices, n, replace=False)

    fig, axes = plt.subplots(n, 5, figsize=(22, 3.5 * n))
    if n == 1: axes = axes[np.newaxis, :]

    col_titles = ['Clean Image', 'Night Input', 'Enhanced (ZeroDCE)',
                  'Quantum Pred', 'Classical Pred']
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontweight='bold', fontsize=10)

    q_model.eval(); c_model.eval()

    for row, idx in enumerate(indices):
        clean_img, label_enc, label_len = clean_ds[idx]
        night_img, _, _                = night_ds[idx]

        true_text = ''.join(IDX2CHAR.get(i.item(),'') for i in label_enc)

        inp_batch = night_img.unsqueeze(0).to(device)
        with torch.no_grad():
            enhanced = q_model.enhancer(inp_batch).cpu().squeeze(0)
            q_pred   = ctc_greedy_decode(q_model(inp_batch).cpu())[0]
            c_pred   = ctc_greedy_decode(c_model(inp_batch).cpu())[0]

        def show(ax, tensor, title=''):
            ax.imshow(np.clip(tensor.permute(1,2,0).numpy(), 0, 1))
            ax.set_title(title, fontsize=8); ax.axis('off')

        show(axes[row, 0], clean_img,   f'True: {true_text}')
        show(axes[row, 1], night_img,   'Night Simulated')
        show(axes[row, 2], enhanced,    'ZeroDCE Output')

        q_ok = (q_pred == true_text)
        c_ok = (c_pred == true_text)
        q_color = '#c8f7c5' if q_ok else '#f7c8c5'
        c_color = '#c8f7c5' if c_ok else '#f7c8c5'

        axes[row, 3].axis('off')
        axes[row, 3].text(0.5, 0.5,
            f"{'✅' if q_ok else '❌'}  {q_pred}\nTrue: {true_text}",
            ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4', fc=q_color, ec='gray'))

        axes[row, 4].axis('off')
        axes[row, 4].text(0.5, 0.5,
            f"{'✅' if c_ok else '❌'}  {c_pred}\nTrue: {true_text}",
            ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4', fc=c_color, ec='gray'))

    plt.suptitle('Quantum ⚡ vs Classical 🔷 — Test Sample Predictions',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    out_path = os.path.join(PROJECT_PATH, 'sample_predictions.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved sample_predictions.png')

show_sample_predictions(q_model, c_model, test_set, n=10)

## Cell 10 — Error Analysis: Where does each model fail?

In [ ]:
# Analyze character-level confusion — which chars does each model get wrong most?
from collections import Counter

def char_confusion(preds, truths, model_name):
    confusion = Counter()
    for pred, true in zip(preds, truths):
        for i, (p, t) in enumerate(zip(pred, true)):
            if p != t:
                confusion[(t, p)] += 1   # (true_char → predicted_char)
    print(f'\n  [{model_name}] Top-10 character confusions (true→pred):')
    for (t,p), cnt in confusion.most_common(10):
        print(f'    {t} → {p}  ({cnt}x)')

char_confusion(q_results_night['preds'], q_results_night['truths'], 'Quantum ⚡')
char_confusion(c_results_night['preds'], c_results_night['truths'], 'Classical 🔷')

print('\n✅ Notebook 2 complete! Your evaluation data is ready for the report.')